In [4]:
import pandas as pd
import plotly.express as px

# eval1
df1 = pd.read_csv('../data/guideline-alignement-eval1.csv', sep=',')

# eval2
df2 = pd.read_csv('../data/guideline-alignement-eval2.csv', sep=',')

# merged
df3 = pd.read_csv('../data/guideline-alignement-merged-eval1-eval2.csv', sep=',')

datasets = [
    ('figure11-a', df1),
    ('figure11-b', df2),
    ('figure11-c', df3),
]

colours = ['#9ecae1', '#efedf5', '#bcbddc', '#756bb1', '#deebf7', '#3182bd', '#e5f5e0', '#a1d99b', '#31a354']

for figure_name, df in datasets:
    df = df.copy()
    df.columns = df.columns.str.strip()

    # Only guidelines with occurrences > 0
    df_filtered = df[df['Occurrences'] > 0].copy()

    # Convert numeric columns
    cols_to_convert = ['Fully aligned', 'Partially aligned', 'Not aligned']
    df_filtered[cols_to_convert] = df_filtered[cols_to_convert].apply(pd.to_numeric, errors='coerce')

    # Calculate mean alignment score
    df_filtered['Mean'] = (
        df_filtered['Fully aligned'] * 3 +
        df_filtered['Partially aligned'] * 2 +
        df_filtered['Not aligned'] * 1
    ) / df_filtered['Occurrences']

    # Ordering by guideline number (extracting numeric part for correct sorting)
    df_filtered = df_filtered.sort_values('Code', key=lambda x: x.str.extract('(\\d+)', expand=False).astype(int))

    fig = px.scatter(
        df_filtered,
        x='Code',
        y='Mean',
        size='Occurrences',
        color='Mean',
        text='Occurrences',
        color_continuous_scale=colours,
        size_max=40,
        title='',
        labels={'Code': 'Guideline', 'Mean': 'Mean alignment score'}
    )

    fig.update_traces(
        textfont_size=14,
        textposition='middle center',
    )

    fig.update_layout(
        xaxis_title='Guideline',
        yaxis_title='Mean alignment score',
        yaxis=dict(
            rangemode='tozero',
            dtick=0.5
        ),
        template='simple_white',
        coloraxis=dict(
            cmin=0,
            cmax=3,
            colorbar=dict(title='')
        ),
        height=600,
        width=1000
    )

    fig.write_image(f'../images/{figure_name}.png', scale=3, width=1000, height=600)
    fig.show()